In [126]:
# Imports, preprocessing config
import numpy as np
import pandas as pd
import gc, json, time
from pathlib import Path
from scipy.ndimage import median_filter
from scipy.signal import find_peaks
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt

CFG = {
    # Preprocessing settings
    'input_dir':  Path('./Test signals from previous years/Suction data/Original'),
    'output_dir': Path('./Test signals from previous years/Suction data/Preprocessed'),
    'original_fs': 50_000,     # Hz
    'target_fs':   500,        # Hz
    'adc_column':  'adc2',
    'adc_col_index': None,
    'chunk_rows': 10_000_000,
    'baseline_sec': 5.0,
    'uint16_offset': 32_768,
    'overwrite': False,
}
CFG['pool_block'] = CFG['original_fs'] // CFG['target_fs']
CFG['output_dir'].mkdir(parents=True, exist_ok=True)

In [127]:
# Cell 2: utility funcions
def sniff_csv(path):
    """Return (delimiter, has_header, n_columns) from the first line."""
    with open(path, 'r') as f:
        first = f.readline()
    delim = '\t' if first.count('\t') >= first.count(',') else ','
    parts = [p.strip() for p in first.rstrip('\r\n').split(delim) if p != '']
    is_numeric = all(p.lstrip('-').replace('.', '', 1).isdigit() for p in parts)
    return delim, (not is_numeric), len(parts)

def load_adc_chunks(path, cfg):
    """Yield the ADC column as int32 arrays, one chunk at a time."""
    delim, header, ncols = sniff_csv(path)
    if header:
        cols = pd.read_csv(path, sep=delim, nrows=0).columns.tolist()
        usecols = cfg['adc_column'] if cfg['adc_column'] in cols else cols[cfg['adc_col_index']]
    else:
        usecols = cfg['adc_col_index']

    for chunk in pd.read_csv(path, sep=delim, usecols=[usecols],
                             header=0 if header else None,
                             chunksize=cfg['chunk_rows'],
                             dtype=np.int32, engine='c'):
        yield chunk.iloc[:, 0].to_numpy(dtype=np.int32)
        del chunk

In [128]:
# Cell 3: Block-mean downsampling (no cross-chunk state, no edge artifacts)
def pool_downsample(chunks, block):
    """Mean-pool every `block` raw samples → 1 output sample."""
    parts, carry = [], np.empty(0, dtype=np.int32)
    total_raw = 0
    for arr in chunks:
        total_raw += len(arr)
        arr = np.concatenate([carry, arr]) if len(carry) else arr
        n_blocks = len(arr) // block
        if n_blocks:
            usable = n_blocks * block
            parts.append(arr[:usable].reshape(n_blocks, block).mean(axis=1, dtype=np.int32))
            carry = arr[usable:].copy()
        else:
            carry = arr
        del arr
    if len(carry):
        parts.append(np.array([carry.mean()], dtype=np.int32))
    return np.concatenate(parts), total_raw

In [129]:
# Cell 4: Full single-file preprocessing (decimate → baseline → uint16 .bin + sidecar)
def preprocess_file(csv_path, cfg):
    bin_path  = cfg['output_dir'] / (csv_path.stem + '.bin')
    meta_path = cfg['output_dir'] / (csv_path.stem + '.meta.json')

    if bin_path.exists() and not cfg['overwrite']:
        print(f"{csv_path.name}: .bin already exists (set overwrite=True to redo)")
        return bin_path

    t0 = time.time()
    raw_mb = csv_path.stat().st_size / 1024**2

    # 1. Read + downsample
    pooled, n_raw = pool_downsample(load_adc_chunks(csv_path, cfg), cfg['pool_block'])
    expected = n_raw / cfg['pool_block']
    assert abs(len(pooled) - expected) <= 1, f"samples lost: {len(pooled)} vs {expected:.1f}"

    # 2. Baseline removal
    win = int(cfg['baseline_sec'] * cfg['target_fs'])
    win += 1 if win % 2 == 0 else 0
    centered = pooled - median_filter(pooled, size=win)

    # 3. Convert to uint16
    u16 = np.clip(centered + cfg['uint16_offset'], 0, 65_535).astype(np.uint16)
    u16.tofile(bin_path)

    # Sidecar info
    meta = {
        'source_file': csv_path.name,
        'original_size_mb': round(raw_mb, 2),
        'raw_samples': n_raw,
        'raw_fs': cfg['original_fs'],
        'target_fs': cfg['target_fs'],
        'samples': int(len(u16)),
        'duration_sec': round(len(u16) / cfg['target_fs'], 2),
        'dtype': 'uint16',
        'value_offset': cfg['uint16_offset'],
        'baseline_window_sec': cfg['baseline_sec'],
        'preprocess_time_sec': round(time.time() - t0, 1),
    }
    meta_path.write_text(json.dumps(meta, indent=2))

In [130]:
# Cell 5: Batch run, smallest files first
files = sorted(CFG['input_dir'].glob('*.csv'), key=lambda p: p.stat().st_size)
print(f"{len(files)} files, smallest → largest:")
for f in files:
    print(f"   {f.stat().st_size/1024**2:8.1f} MB  {f.name}")
print()

for csv_path in files:
    try:
        preprocess_file(csv_path, CFG)
        gc.collect()
    except Exception as e:
        print(f"{csv_path.name}: {e}")

9 files, smallest → largest:
       29.6 MB  8. suction_MIDDLE_CONC_1ml_(and_no_resuction).csv
       30.2 MB  5. suction_HIGH_CONC_1ml_and_no_resuction.csv
       30.9 MB  9. suction_MIDDLE_CONC_1ml_(and_no_resuction (2)).csv
       55.2 MB  6. suction_MIDDLE_CONC_3ml_(and_no_resuction).csv
      126.4 MB  7. suction_MIDDLE_CONC_2ml_(and_1_times_resuction).csv
      182.7 MB  4. suction_HIGH_CONC_1ml_(and_2_times_resuction).csv
      265.1 MB  3. suction_HIGH_CONC_2ml_(and_2_times_resuction.csv
      343.7 MB  2. suction_HIGH_CONC_4ml_(and_2_times_resuction).csv
     7242.4 MB  1. LONG_suction_HIGH_CONC_4ml_(and_3_times_resuction).csv

8. suction_MIDDLE_CONC_1ml_(and_no_resuction).csv: .bin already exists (set overwrite=True to redo)
5. suction_HIGH_CONC_1ml_and_no_resuction.csv: .bin already exists (set overwrite=True to redo)
9. suction_MIDDLE_CONC_1ml_(and_no_resuction (2)).csv: .bin already exists (set overwrite=True to redo)
6. suction_MIDDLE_CONC_3ml_(and_no_resuction).csv: .bin

In [190]:
# Cell 6: Region Detection Configuration
REGION_CONFIG = {
    'data_dir': Path('./Test signals from previous years/Suction data/Preprocessed'),
    'output_dir': Path('./Test signals from previous years/Suction data/Region'),

    # Activity detection thresholds
    'activity_window_samples': 50,      # activity_window_samples*1000/Hz = ms of sliding window, where Hz is the downsampled Hz
    'activity_multiplier': 1.25,         # >activity_multiplier*baseline = active 
    'min_region_duration_sec': 0.1,     # Ignore regions shorter than this (noise)
    'padding_sec': 0,                 # Add buffer before/after each region, in seconds
}
REGION_CONFIG['target_fs'] = CFG['target_fs']
REGION_CONFIG['padding_samples'] = int(REGION_CONFIG['padding_sec'] * CFG['target_fs'])
REGION_CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

In [191]:
# Cell 7: Active Region Detection + Extraction
def detect_regions_and_extract(bin_path, cfg):
    """
    Detect contiguous active regions → store as blocks → save
    Output: filename, time, file size, percent stored
    """
    meta_path = bin_path.with_suffix('.meta.json')
    if not meta_path.exists():
        raise FileNotFoundError(f"No metadata found for {bin_path}")

    meta = json.loads(meta_path.read_text())

    # Memory-map original data
    raw_data = np.memmap(str(bin_path), dtype='uint16', mode='r')
    centered = (raw_data.astype(np.int32) - meta['value_offset'])

    # Compute rolling std to detect activity
    activity_std = pd.Series(centered).rolling(cfg['activity_window_samples'], center=True, min_periods=1).std().to_numpy()
    
    # Baseline is overall std
    baseline_std = np.std(centered)
    activity_threshold = cfg['activity_multiplier'] * baseline_std
    
    # Mark active samples (where local std > threshold)
    active_mask = activity_std > activity_threshold

    # Find contiguous active regions
    regions = []
    in_region = False
    region_start = 0
    
    for i, is_active in enumerate(active_mask):
        if is_active and not in_region:
            region_start = i
            in_region = True
        elif not is_active and in_region:
            region_end = i
            
            # Apply padding
            padded_start = max(0, region_start - cfg['padding_samples'])
            padded_end = min(len(centered), region_end + cfg['padding_samples'])
            
            # Skip too-short regions
            duration_sec = (padded_end - padded_start) / cfg['target_fs']
            if duration_sec >= cfg['min_region_duration_sec']:
                regions.append({
                    'global_start': int(padded_start),
                    'global_end': int(padded_end),
                    'duration_sec': round(duration_sec, 4),
                })
            
            in_region = False

    if in_region:
        padded_start = max(0, region_start - cfg['padding_samples'])
        padded_end = len(centered)
        duration_sec = (padded_end - padded_start) / cfg['target_fs']
        
        if duration_sec >= cfg['min_region_duration_sec']:
            regions.append({
                'global_start': int(padded_start),
                'global_end': int(padded_end),
                'duration_sec': round(duration_sec, 4),
            })

    # Concatenate all region data
    all_region_data = []
    for region in regions:
        segment = centered[region['global_start']:region['global_end']]
        all_region_data.append(segment)

    if len(all_region_data) > 0:
        compressed_regions = np.concatenate(all_region_data).astype(np.uint16) + meta['value_offset']
    else:
        compressed_regions = np.array([], dtype=np.uint16)

    # Calculate stats
    total_stored_samples = len(compressed_regions)
    total_original_samples = meta['samples']
    compression_ratio = total_original_samples / total_stored_samples if total_stored_samples > 0 else 1.0
    total_region_duration = sum(r['duration_sec'] for r in regions)

    # Build output metadata
    output_meta = {
        'source_file': bin_path.name,
        'original_duration_sec': meta['duration_sec'],
        'original_samples': total_original_samples,
        'num_regions': len(regions),
        'stored_regions': regions,
        'total_stored_samples': total_stored_samples,
        'compression_ratio': round(compression_ratio, 2),
        'original_size_mb': meta.get('original_size_mb', 0),
        'stored_size_mb': round(total_stored_samples * 2 / 1024**2, 3),
        'total_region_duration_sec': round(total_region_duration, 2),
        'region_efficiency': round(total_region_duration / meta['duration_sec'], 4),
        'baseline_std': round(baseline_std, 2),
        'activity_threshold': round(activity_threshold, 2),
        'value_offset': meta['value_offset'],
        'target_fs': cfg['target_fs'],
        'zstd_compressed': False,
    }

    # Determine output path
    output_path = cfg['output_dir'] / (bin_path.stem + '_regions.bin')
    meta_path_out = cfg['output_dir'] / (bin_path.stem + '_regions.meta.json')

    # Write data
    if len(compressed_regions) > 0:
        compressed_regions.tofile(output_path)

    # Write metadata
    meta_path_out.write_text(json.dumps(output_meta, indent=2))

    print(f"{output_meta['original_size_mb']:.2f}MB → {output_meta['stored_size_mb']:.2f}MB ({output_meta['region_efficiency']*100:.1f}% stored) | {bin_path.name}")

    return output_meta

In [192]:
# Cell 8: Full Batch Region Extraction (minimal output)
def run_batch_region_extraction():
    """Process all .bin files with region-based extraction (uncompressed)"""
    bin_files = sorted(REGION_CONFIG['data_dir'].glob('*.bin'))

    results = []
    total_original_mb = 0
    total_stored_mb = 0
    total_time = 0

    for i, bin_path in enumerate(bin_files):
        start = time.time()

        try:
            result = detect_regions_and_extract(bin_path, REGION_CONFIG)
            elapsed = time.time() - start
            total_time += elapsed

            total_original_mb += result['original_size_mb']
            total_stored_mb += result['stored_size_mb']

            results.append({
                'file': bin_path.name,
                **result,
                'processing_ms': round(elapsed * 1000, 1),
            })

        except Exception as e:
            print(f"{bin_path.name} | ERROR: {e}")
            results.append({'file': bin_path.name, 'error': str(e)})

    # Summary
    total_ratio = total_original_mb / total_stored_mb if total_stored_mb > 0 else 1
    print(f"{total_original_mb:.2f}MB → {total_stored_mb:.2f}MB ({(total_stored_mb/total_original_mb*100):.1f}% stored) | All files processed in {total_time:.1f}s")

    # Save results
    df = pd.DataFrame(results)
    df.to_csv(REGION_CONFIG['output_dir'] / 'region_extraction_results.csv', index=False)

    return results

In [193]:
# Cell 9: Load Region Data Utility
def load_region_data(region_bin_path):
    """Load region-based data (plain .bin files only)"""
    meta_path = region_bin_path.with_suffix('.meta.json')
    meta = json.loads(meta_path.read_text())

    # Load region data (uncompressed .bin)
    regions_data = np.fromfile(region_bin_path, dtype=np.uint16)

    # Convert back to centered values
    centered = regions_data.astype(np.int32) - meta['value_offset']

    # Reconstruct individual regions
    regions = []
    idx = 0
    for region_info in meta['stored_regions']:
        region_size = region_info['global_end'] - region_info['global_start']
        region_data = centered[idx:idx + region_size]
        regions.append({
            'timestamp_sec': region_info['global_start'] / meta['target_fs'],
            'data': region_data,
            'duration_sec': region_info['duration_sec'],
        })
        idx += region_size

    return {
        'regions': regions,
        'num_regions': len(regions),
        'meta': meta,
    }

In [194]:
# Cell 10: Run region extraction
results = run_batch_region_extraction()

7242.40MB → 0.21MB (1.3% stored) | 1. LONG_suction_HIGH_CONC_4ml_(and_3_times_resuction).bin
343.74MB → 0.04MB (5.6% stored) | 2. suction_HIGH_CONC_4ml_(and_2_times_resuction).bin
265.06MB → 0.04MB (6.3% stored) | 3. suction_HIGH_CONC_2ml_(and_2_times_resuction.bin
182.69MB → 0.02MB (5.2% stored) | 4. suction_HIGH_CONC_1ml_(and_2_times_resuction).bin
30.22MB → 0.00MB (5.8% stored) | 5. suction_HIGH_CONC_1ml_and_no_resuction.bin
55.20MB → 0.00MB (3.0% stored) | 6. suction_MIDDLE_CONC_3ml_(and_no_resuction).bin
126.44MB → 0.01MB (3.6% stored) | 7. suction_MIDDLE_CONC_2ml_(and_1_times_resuction).bin
29.59MB → 0.00MB (0.5% stored) | 8. suction_MIDDLE_CONC_1ml_(and_no_resuction).bin
30.91MB → 0.00MB (4.3% stored) | 9. suction_MIDDLE_CONC_1ml_(and_no_resuction (2)).bin
8306.25MB → 0.33MB (0.0% stored) | All files processed in 0.9s


In [195]:
# Cell 11: Peak Detection Configuration
PEAK_CONFIG = {
    'results_dir': Path('./Test signals from previous years/Suction data/Results'),
    
    # Peak detection parameters
    'min_peak_height': 75,           # Minimum height above baseline (in centered units)
    'min_distance_samples': 250,     # min_distance_samples/Hz = seconds between peaks, where Hz is the downsampled Hz
    'prominence': 30,                # Peak prominence threshold
}
PEAK_CONFIG['results_dir'].mkdir(parents=True, exist_ok=True)

In [196]:
# Cell 12: Peak Detection Function
def detect_peaks_in_regions(region_data_result, cfg):
    """
    Detect peaks in each loaded region using scipy.signal.find_peaks
    
    Returns: list of dicts with peak info for each region
    """
    all_peaks = []
    regions = region_data_result['regions']
    fs = region_data_result['meta']['target_fs']
    
    for reg_idx, region in enumerate(regions):
        data = region['data']
        time_axis = np.arange(len(data)) / fs
        
        # Use scipy's find_peaks with configured parameters
        peaks, properties = find_peaks(
            data,
            height=cfg['min_peak_height'],
            distance=cfg['min_distance_samples'],
            prominence=cfg['prominence']
        )
        
        # Store peak information
        region_peaks = {
            'region_index': reg_idx,
            'region_start_sec': region['timestamp_sec'],
            'num_peaks': len(peaks),
            'peak_times_sec': (time_axis[peaks] + region['timestamp_sec']).tolist(),
            'peak_values': data[peaks].tolist(),
            'local_times_sec': time_axis[peaks].tolist(),
        }
        all_peaks.append(region_peaks)
    
    return all_peaks

In [197]:
# Cell 13: Batch Peak Detection for All Files
def run_batch_peak_detection():
    """Process all region files and save peak detection results"""
    region_files = sorted(REGION_CONFIG['output_dir'].glob('*_regions.bin'))
    
    all_results = []
    
    for region_bin_path in region_files:
        try:
            print(f"Processing: {region_bin_path.name}")
            region_result = load_region_data(region_bin_path)
            peaks = detect_peaks_in_regions(region_result, PEAK_CONFIG)
            
            # Aggregate peak counts per file
            file_total_peaks = sum(p['num_peaks'] for p in peaks)
            file_meta = region_result['meta']
            
            all_results.append({
                'file': region_bin_path.name,
                'num_regions': region_result['num_regions'],
                'total_peaks': file_total_peaks,
                'original_duration_sec': file_meta['original_duration_sec'],
                'region_efficiency': file_meta['region_efficiency'],
            })
            
            # Save individual peak metadata
            peak_meta_path = REGION_CONFIG['output_dir'] / (region_bin_path.stem + '_peaks.meta.json')
            meta_output = {
                **file_meta,
                'detected_peaks': peaks,
                'total_peaks': file_total_peaks,
            }
            peak_meta_path.write_text(json.dumps(meta_output, indent=2))
            
        except Exception as e:
            print(f"{region_bin_path.name} | ERROR: {e}")
            all_results.append({'file': region_bin_path.name, 'error': str(e)})
    
    # Save batch summary
    df_summary = pd.DataFrame(all_results)
    df_summary.to_csv(REGION_CONFIG['output_dir'] / 'peak_detection_summary.csv', index=False)
    
    print(f"\nBatch complete! Processed {len(region_files)} files.")
    print(df_summary[['file', 'num_regions', 'total_peaks']].to_string())
    
    return all_results

batch_results = run_batch_peak_detection()

Processing: 1. LONG_suction_HIGH_CONC_4ml_(and_3_times_resuction)_regions.bin
Processing: 2. suction_HIGH_CONC_4ml_(and_2_times_resuction)_regions.bin
Processing: 3. suction_HIGH_CONC_2ml_(and_2_times_resuction_regions.bin
Processing: 4. suction_HIGH_CONC_1ml_(and_2_times_resuction)_regions.bin
Processing: 5. suction_HIGH_CONC_1ml_and_no_resuction_regions.bin
Processing: 6. suction_MIDDLE_CONC_3ml_(and_no_resuction)_regions.bin
Processing: 7. suction_MIDDLE_CONC_2ml_(and_1_times_resuction)_regions.bin
Processing: 8. suction_MIDDLE_CONC_1ml_(and_no_resuction)_regions.bin
Processing: 9. suction_MIDDLE_CONC_1ml_(and_no_resuction (2))_regions.bin

Batch complete! Processed 9 files.
                                                                file  num_regions  total_peaks
0  1. LONG_suction_HIGH_CONC_4ml_(and_3_times_resuction)_regions.bin         1388         1362
1       2. suction_HIGH_CONC_4ml_(and_2_times_resuction)_regions.bin          344          340
2        3. suction_HIGH_CON

In [198]:
# Cell 17: Full Signal Visualization with Peaks (CORRECTED - Region folder)
def plot_full_signal_with_peaks(bin_path, cfg=PEAK_CONFIG, output_dir=None):
    """Plot complete signal with peaks marked across entire duration"""
    if output_dir is None:
        output_dir = PEAK_CONFIG['results_dir']  # Results/ for images
    
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Load full preprocessed data from Preprocessed/
    meta_path = bin_path.with_suffix('.meta.json')
    meta = json.loads(meta_path.read_text())
    
    raw_data = np.memmap(str(bin_path), dtype='uint16', mode='r')
    centered = (raw_data.astype(np.int32) - meta['value_offset'])
    
    # FIXED: Look for peak files in Region/ folder (where they actually are)
    peak_meta_name = f"{bin_path.stem}_regions_peaks.meta.json"
    peak_meta_path = REGION_CONFIG['output_dir'] / peak_meta_name  # ← Changed to Region/
    
    # Create plot
    fig, ax = plt.subplots(figsize=(16, 4))
    time_axis = np.arange(len(centered)) / meta['target_fs']
    
    ax.plot(time_axis, centered, 'b-', linewidth=0.3, alpha=0.6, label='Signal')
    
    # Map and mark peaks
    peak_count = 0
    if peak_meta_path.exists():
        peak_meta = json.loads(peak_meta_path.read_text())
        detected_peaks = peak_meta.get('detected_peaks', [])
        
        all_peak_times = []
        all_peak_values = []
        
        for region_peak in detected_peaks:
            region_start = region_peak['region_start_sec']
            local_times = region_peak['local_times_sec']
            peak_vals = region_peak['peak_values']
            
            for t, v in zip(local_times, peak_vals):
                global_t = region_start + t
                all_peak_times.append(global_t)
                all_peak_values.append(v)
                peak_count += 1
        
        if len(all_peak_times) > 0:
            ax.scatter(all_peak_times, all_peak_values, 
                      c='red', s=8, zorder=5, marker='^',
                      edgecolors='darkred', linewidths=0.3,
                      label=f'Detected peaks (n={len(all_peak_times)})')
        print(f"  ✅ Found peaks: {peak_count}")
    else:
        print(f"  ⚠️ Peak metadata not found: {peak_meta_name}")
    
    ax.set_xlabel('Time (seconds)', fontsize=10)
    ax.set_ylabel('Amplitude (centered)', fontsize=10)
    ax.set_title(f"{bin_path.name}\nDuration: {meta['duration_sec']:.1f}s | {meta['samples']:,} samples | {peak_count} peaks", 
                fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='upper right', fontsize=9)
    
    plt.tight_layout()
    
    output_path = output_dir / (bin_path.stem + '_full_signal.png')
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    return output_path

In [199]:
# Cell 18: Batch process all files
def run_full_batch_visualization():
    bin_files = sorted(CFG['output_dir'].glob('*.bin'))
    
    print(f"Peak files in: {REGION_CONFIG['output_dir']}")
    print(f"Images will be saved to: {PEAK_CONFIG['results_dir']}")
    print()
    
    plot_paths = []
    for i, bin_path in enumerate(bin_files):
        try:
            plot_path = plot_full_signal_with_peaks(bin_path)
            plot_paths.append(plot_path)
        except Exception as e:
            print(f"ERROR: {e}")
    
    print(f"\nDone! Generated {len(plot_paths)} plots")
    return plot_paths

all_plots = run_full_batch_visualization()

Peak files in: Test signals from previous years/Suction data/Region
Images will be saved to: Test signals from previous years/Suction data/Results

  ✅ Found peaks: 1362
  ✅ Found peaks: 340
  ✅ Found peaks: 287
  ✅ Found peaks: 170
  ✅ Found peaks: 29
  ✅ Found peaks: 28
  ✅ Found peaks: 72
  ✅ Found peaks: 3
  ✅ Found peaks: 21

Done! Generated 9 plots
